<a href="https://colab.research.google.com/github/s251198/CS521_das44_HW4/blob/main/RL_IN_LLMS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install -q "lm-eval[hf]"
%pip install transformers pandas
%pip install -q langdetect

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
# definning the four OLMo checkpoint models

MODELS = {
    "Base": "allenai/OLMo-2-0425-1B",
    "SFT": "allenai/OLMo-2-0425-1B-SFT",
    "DPO": "allenai/OLMo-2-0425-1B-DPO",
    "Instruct": "allenai/OLMo-2-0425-1B-Instruct"
}

BENCHMARKS = ["gsm8k", "ifeval"]

num_examples = 200
batch_size = 4
seed = 42

import os

output_dir = "olmo_results"
os.makedirs(output_dir, exist_ok = True)

for model_name, model_path in MODELS.items():
    print(f"Model: {model_name}, Path: {model_path}")

print("\nBenchmarks:", BENCHMARKS)
print("Examples per benchmark:", num_examples)

Model: Base, Path: allenai/OLMo-2-0425-1B
Model: SFT, Path: allenai/OLMo-2-0425-1B-SFT
Model: DPO, Path: allenai/OLMo-2-0425-1B-DPO
Model: Instruct, Path: allenai/OLMo-2-0425-1B-Instruct

Benchmarks: ['gsm8k', 'ifeval']
Examples per benchmark: 200


In [ ]:
# Checking if the tokenizer has a chat template
from transformers import AutoTokenizer

for model_name, model_path in MODELS.items():
    print("\nChecking:", model_name)
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    has_template = tokenizer.chat_template is not None
    print("Chat template available:", has_template)

    if model_name != "Base" and not has_template:
        raise ValueError(
            f"{model_name} does not have a chat template. "
            "Check the checkpoint tokenizer configuration."
        )

print("\nAll tokenizer checks completed.")



Checking: Base
Chat template available: False

Checking: SFT
Chat template available: True

Checking: DPO
Chat template available: True

Checking: Instruct
Chat template available: True

All tokenizer checks completed.


In [ ]:
# Running the evaluations
import subprocess

RESULT_FILES = {}

for model_name, model_id in MODELS.items():

    RESULT_FILES[model_name] = {}

    for benchmark in BENCHMARKS:

        print("\n" + "=" * 60)
        print("Model:", model_name)
        print("Benchmark:", benchmark)
        print("=" * 60)

        output_file = os.path.join(
            output_dir,
            f"{model_name}_{benchmark}.json"
        )

        command = [
            "lm_eval",
            "--model", "hf",
            "--model_args",
            f"pretrained={model_id},dtype=float16,trust_remote_code=True",
            "--tasks", benchmark,
            "--num_fewshot", "0",
            "--batch_size", str(batch_size),
            "--device", "cuda:0",
            "--limit", str(num_examples),
            "--seed", str(seed),
            "--gen_kwargs",
            "temperature=0.0,do_sample=False",
            "--log_samples",
            "--output_path", output_file
        ]

        # Apply the model's chat template to tuned checkpoints
        if model_name in ["SFT", "DPO", "Instruct"]:
            command.append("--apply_chat_template")

        result = subprocess.run(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True
        )

        print(result.stdout)

        if result.returncode != 0:
            raise RuntimeError(
                f"Failed: {model_name} - {benchmark}"
            )

        RESULT_FILES[model_name][benchmark] = output_file

        print("Saved:", output_file)

print("\nAll eight evaluations completed!")



Model: Base
Benchmark: gsm8k
2026-11-10:07:23:45 WARNING  [config.evaluate_config:337] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-11-10:07:24:05 INFO     [_cli.run:392] Selected Tasks: ['gsm8k']
2026-11-10:07:24:07 INFO     [evaluator:214] Setting random seed to 42 | Setting numpy seed to 42 | Setting torch manual seed to 42 | Setting fewshot manual seed to 42
2026-11-10:07:24:07 WARNING  [evaluator:226] generation_kwargs: {'temperature': 0.0, 'do_sample': False} specified through cli, these settings will update set parameters in yaml tasks. Ensure 'do_sample=True' for non-greedy decoding!
2026-11-10:07:24:07 INFO     [evaluator:239] Initializing hf model, with arguments: {'pretrained': 'allenai/OLMo-2-0425-1B', 'dtype': 'float16', 'trust_remote_code': True}
2026-11-10:07:24:11 INFO     [models.huggingface:289] Using device 'cuda:0'
2026-11-10:07:24:12 INFO     [models.huggingface:582] Model parallel was set to False, max memory was 

In [ ]:

import os
import glob
import json
import pandas as pd

results = []

for model_name in MODELS.keys():
    for benchmark in BENCHMARKS:

        pattern = f"olmo_results/{model_name}_{benchmark}_*.json"
        matching_files = glob.glob(pattern)

        if not matching_files:
            print(f"Missing result: {model_name} - {benchmark}")
            continue

        result_file = max(matching_files, key=os.path.getmtime)

        with open(result_file, "r") as f:
            data = json.load(f)

        metrics = data["results"][benchmark]

        if benchmark == "gsm8k":
            score = metrics.get("exact_match,flexible-extract")

            results.append({
                "Model": model_name,
                "Benchmark": "GSM8K",
                "Accuracy (%)": score * 100 if score is not None else None
            })

        elif benchmark == "ifeval":
            strict = metrics.get("prompt_level_strict_acc,none")
            loose = metrics.get("prompt_level_loose_acc,none")

            results.append({
                "Model": model_name,
                "Benchmark": "IFEval",
                "Accuracy (%)": None,
                "Strict Accuracy (%)": strict * 100 if strict is not None else None,
                "Loose Accuracy (%)": loose * 100 if loose is not None else None
            })

results_df = pd.DataFrame(results)

display(results_df.round(2))


,Model,Benchmark,Accuracy (%),Strict Accuracy (%),Loose Accuracy (%)
0,Base,GSM8K,4.0,NaN,NaN
1,Base,IFEval,NaN,9.5,11.0
2,SFT,GSM8K,47.5,NaN,NaN
3,SFT,IFEval,NaN,41.0,44.5
4,DPO,GSM8K,46.5,NaN,NaN
5,DPO,IFEval,NaN,55.5,59.0
6,Instruct,GSM8K,67.0,NaN,NaN
7,Instruct,IFEval,NaN,58.0,64.0


In [ ]:
# displaying the examples with questions and teh answers.
import os
import glob
import json

def display_examples(model_name, benchmark, count=2):

    result_pattern = (
        f"olmo_results/{model_name}_{benchmark}_*.json"
    )

    result_files = glob.glob(result_pattern)

    if not result_files:
        print(f"No results found for {model_name} - {benchmark}")
        return

    result_file = max(result_files, key=os.path.getmtime)

    # Matching the sample file using the same timestamp
    timestamp = os.path.basename(result_file)
    timestamp = timestamp[len(f"{model_name}_{benchmark}_"):]
    timestamp = timestamp[:-5]

    sample_file = os.path.join(
        "olmo_results",
        f"samples_{benchmark}_{timestamp}.jsonl"
    )

    if not os.path.exists(sample_file):
        print(f"No sample file found: {sample_file}")
        return

    print("\n" + "=" * 75)
    print(f"{model_name} - {benchmark}")
    print("=" * 75)

    with open(sample_file, "r", encoding="utf-8") as f:
        samples = [json.loads(line) for line in f if line.strip()]

    for i, sample in enumerate(samples[:count]):

        print(f"\nExample {i + 1}")

        doc = sample.get("doc", {})

        if isinstance(doc, dict):
            question = doc.get(
                "question",
                doc.get("prompt", str(doc))
            )
        else:
            question = doc

        print("\nQuestion / Instruction:")
        print(question)

        print("\nGenerated response:")
        print(sample.get("resps", "Not available"))

        print("\nExpected answer:")
        print(sample.get("target", "Not available"))

        print("-" * 75)


# Display two examples for every checkpoint and benchmark
for model_name in MODELS:
    display_examples(model_name, "gsm8k", count=2)
    display_examples(model_name, "ifeval", count=2)



Base - gsm8k

Example 1

Question / Instruction:
Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

Generated response:
[[' 16 eggs x $2 = $32\n']]

Expected answer:
Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.
#### 18
---------------------------------------------------------------------------

Example 2

Question / Instruction:
A robe takes 2 bolts of blue fiber and half that much white fiber.  How many bolts in total does it take?

Generated response:
[[' 4']]

Expected answer:
It takes 2/2=<<2/2=1>>1 bolt of white fiber
So the total amount of fabric is 2+1=<<2+1=3>>3 bolts of fabric
#### 3
---------------------------------------------------------------------------

Base - ifeval
